In [1]:
import dask
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.patches as patches
import numpy as np
import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmcrameri as cm
import os
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.patches as patches
from matplotlib.lines import Line2D
import matplotlib.patheffects as pe
import warnings
from cmcrameri import cm
import cfgrib
import glob

import babet as bb
import atmicp

warnings.filterwarnings("ignore", category=FutureWarning)
dask.config.set(**{'array.slicing.split_large_chunks': True})

In [2]:
atmicp.style.set_style()

# climate scenario colours from the shared style
CLIM = {'pi': atmicp.style.climate_color('1870'),        # blue
        'curr': atmicp.style.climate_color('present'),   # grey
        'incr': atmicp.style.climate_color('future1')}   # orange

LABEL = {'pi': 'pre-industrial', 'curr': 'current', 'incr': 'increased'}

# one linestyle per perturbation, so climate stays the only thing colour encodes
LS = {'iterated': '-.',
      'iterated_short': ':',
      'tq': '--',
      'ocean': '-',
      'none': '-'}

PERT_LABEL = {'iterated': 'iterated',
              'iterated_short': 'iterated, short',
              'tq': 'delta t+q',
              'ocean': 'ocean only',
              'none': 'unperturbed'}

In [8]:
## and one for the z indices

def extract_z_indices(ds):
    
    # get three indices of geopotential height - mean, max & L-O blocking
    z_max = ds.z.sel(latitude=slice(60,35),longitude=slice(230,250)).max(['latitude','longitude'])
    z_block = (ds.z.sel(latitude=40,longitude=slice(230,250)) - ds.z.sel(latitude=60,longitude=slice(230,250))).mean('longitude')
    z_mean = ds.z.sel(latitude=slice(60,35),longitude=slice(230,250)).weighted(np.cos(np.deg2rad(ds.latitude))).mean(['latitude','longitude'])
    
    return xr.merge([z_max.rename('z500_max'),z_mean.rename('z500_mean'),z_block.rename('z500_blo')])

# Import data

In [3]:
# tcc and swvl1 on top of the defaults, for the corner plot
ifs_sfc = atmicp.Data.get_ifs_data(levtype='sfc', res='US025',
                                   variables=['t2m', 'msl', 'tcwv', 'tcc', 'swvl1'])

Loading 3 file(s) for b2vj (cf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/pi/US025/sfc/cf
Loading 3 file(s) for b2vj (pf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/pi/US025/sfc/pf
Loading 3 file(s) for b2vr (cf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/pi/US025/sfc/cf
Loading 3 file(s) for b2vr (pf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/pi/US025/sfc/pf
Loading 3 file(s) for b2vl (cf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/pi/US025/sfc/cf
Loading 3 file(s) for b2vl (pf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/pi/US025/sfc/pf
Loading 3 file(s) for b2vo (cf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/pi/US025/sfc/cf
Loading 3 file(s) for b2vo (pf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/pi/US025/sfc/pf
Loading 3 file(s) for b2ut (cf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/curr/US025/sfc/cf
Loading 3 file(s) for b2ut (pf) from /gf5/pr

In [4]:
ifs_pl = atmicp.Data.get_ifs_data(levtype='pl', res='GLO100')

Loading 3 file(s) for b2vj (cf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/pi/GLO100/pl/cf
Loading 3 file(s) for b2vj (pf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/pi/GLO100/pl/pf
Loading 3 file(s) for b2vr (cf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/pi/GLO100/pl/cf
Loading 3 file(s) for b2vr (pf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/pi/GLO100/pl/pf
Loading 3 file(s) for b2vl (cf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/pi/GLO100/pl/cf
Loading 3 file(s) for b2vl (pf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/pi/GLO100/pl/pf
Loading 3 file(s) for b2vo (cf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/pi/GLO100/pl/cf
Loading 3 file(s) for b2vo (pf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/pi/GLO100/pl/pf
Loading 3 file(s) for b2ut (cf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/curr/GLO100/pl/cf
Loading 3 file(s) for b2ut (pf) from /gf5/pr

/home/e/ermis/ATMICP/atmicp/data.py:255: UserWarning: b2vq: missing inidates ['b2vq_2021-06-22.nc', 'b2vq_2021-06-26.nc'] in /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/incr/GLO100/pl/cf
  warnings.warn(
/home/e/ermis/ATMICP/atmicp/data.py:255: UserWarning: b2vq: missing inidates ['b2vq_2021-06-22.nc', 'b2vq_2021-06-26.nc'] in /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/incr/GLO100/pl/pf
  warnings.warn(


Loading 3 file(s) for b2vm (pf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/incr/GLO100/pl/pf
Loading 3 file(s) for b2vn (cf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/incr/GLO100/pl/cf
Loading 3 file(s) for b2vn (pf) from /gf5/predict/AWH019_ERMIS_ATMICP/ITERATION/MED-R/EXP/incr/GLO100/pl/pf


In [5]:
era5_sfc = xr.open_mfdataset(glob.glob('/network/group/aopp/archive02a/projects/predict/AWH009_LEACH_RELIABLE/COUNTERFACTUAL-FORECASTING/DATA/postproc/WWAmean/era5/sfc/*.nc'))
# convert variables to consistent units:
era5_sfc['tp'] *= 24 * 1000 # convert to mm day-1
era5_sfc['e'] *= 1 / 3600 # convert to m s-1
print('got ERA5')

got ERA5


In [9]:
era5_z500 = extract_z_indices(xr.open_mfdataset(glob.glob('/network/group/aopp/archive02a/projects/predict/AWH009_LEACH_RELIABLE/COUNTERFACTUAL-FORECASTING/DATA/era5/glo100/pl/*.nc'))).load()
print('got ERA5')

got ERA5


In [7]:
# q, u and v for IVT are in the rquv file; the dztw file next to it only has d, z, t and w
era5_glo_pl = xr.open_dataset('/network/group/aopp/archive02a/projects/predict/AWH009_LEACH_RELIABLE/COUNTERFACTUAL-FORECASTING/DATA/era5/glo250/pl/2021_MJJA_rquv.nc')
print('got ERA5')

got ERA5


# Process data

In [10]:
pnw = atmicp.Constants.pnw
box = dict(longitude=slice(pnw[0], pnw[1]), latitude=slice(pnw[3], pnw[2]))
ifs_pnw = (ifs_sfc.sel(time=slice('2021-06-26', '2021-07-01'), **box)
           .mean(['latitude', 'longitude']).compute())

# climate/perturbation combinations that never ran are NaN-filled by the concat
ifs_pnw = ifs_pnw.stack(z=['perturbation', 'climate', 'inidate', 'number']).dropna('z', how='all')

max_t2m_times = ifs_pnw.t2m.idxmax('time')
exp_max_df = ifs_pnw.sel(time=max_t2m_times).to_dataframe()

# Compare IVT locations in time